# Stability Analysis: Topics, Random Seeds, and Evaluation Metrics

This tutorial systematically evaluates the stability and performance of scE2TM across different configurations.

## What this tutorial covers

- **Topic numbers**: 10, 20, 50, 80, 100
- **Random seeds**: 1, 2, 3, 4, 5 (for reproducibility assessment)
- **Training epochs**: 500 (full training)

## Evaluation metrics

For each configuration, we compute:

- **Topic Diversity (TD)**: Measures the uniqueness of top genes across topics. Higher values indicate more diverse topics.
- **Topic Coherence (TC)**: Measures the semantic consistency of top genes using MSigDB pathway annotations. Higher values indicate more biologically meaningful topics.
- **ARI / NMI**: Clustering performance metrics comparing learned cell embeddings with ground truth cell type labels (requires labels).
- **Topic Stability**: Cosine similarity between topics learned from different random seeds after Hungarian matching. Measures how robust the learned topics are to random initialization.



In [ ]:
import os
import time
import pandas as pd
import numpy as np
from pathlib import Path
from scE2TM import scE2TM

# ========== Configuration ==========
# Set this to True for labeled mode, False for label-free mode
USE_LABELS = False  # Change to False for label-free mode

# Other configurations
DATASET_NAME = 'Wang'
TOPICS = [10, 20, 50, 80, 100]
# SEEDS = [1, 2, 3, 4, 5]
# EPOCHS = 500  # Full training
SEEDS = [1, 2]  # quick testing
EPOCHS = 100
GPU_ID = 0

# ========== Setup paths ==========
# Manually set project root (adjust this path if needed)
PROJECT_ROOT = Path('/PATH/scE2TM')

# Change working directory to project root
os.chdir(PROJECT_ROOT)
print(f"Working directory set to: {os.getcwd()}")

DATA_DIR = PROJECT_ROOT / 'data'
# Use DATASET_NAME variable instead of hardcoding 'Wang'
BASE_OUTPUT_DIR = PROJECT_ROOT / 'output' / DATASET_NAME

print(f"Data directory: {DATA_DIR}")
print(f"Data directory exists: {DATA_DIR.exists()}")
print(f"Output directory: {BASE_OUTPUT_DIR}")

# Verify msigdb file exists
msigdb_path = DATA_DIR / 'msigdb.v2024.1.Hs.symbols.gmt'
if msigdb_path.exists():
    print(f"✓ MSigDB file found")
else:
    print(f"Warning: MSigDB file not found at {msigdb_path}")
    print("Topic coherence calculation may fail.")

# Create base output directory
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ========== Check label file if in labeled mode ==========
if USE_LABELS:
    label_file = DATA_DIR / f'{DATASET_NAME}_cell_anno.csv'
    if not label_file.exists():
        print(f"\nERROR: Label file not found at: {label_file}")
        print(f"Files currently in {DATA_DIR}:")
        for f in DATA_DIR.glob('*'):
            print(f"  - {f.name}")
        raise FileNotFoundError(
            f"Label file not found. Please ensure {label_file} exists, "
            f"or set USE_LABELS = False to run without labels."
        )
    print(f"✓ Label file found: {label_file}")

# ========== Helper function to run experiment ==========
def run_experiment(num_topics, seed, use_labels, subdir):
    """Run a single experiment and return results"""
    
    print(f"  Topics: {num_topics}, Seed: {seed}, Labels: {use_labels}")
    
    start_time = time.time()
    
    try:
        results = scE2TM(
            dataset_name=DATASET_NAME,
            data_dir=str(DATA_DIR),
            output_dir=str(PROJECT_ROOT / 'output'),
            num_topics=num_topics,
            epochs=EPOCHS,
            gpu_id=GPU_ID,
            use_labels=use_labels,
            seed=seed,
            subdir=subdir
        )
        
        elapsed_time = time.time() - start_time
        
        # Extract metrics from results (new api returns these)
        result_entry = {
            'topics': num_topics,
            'seed': seed,
            'use_labels': use_labels,
            'time_min': round(elapsed_time / 60, 2),
            'status': 'success',
            'subdir': subdir,
            # Evaluation metrics
            'topic_diversity': results.get('topic_diversity'),
            'topic_coherence': results.get('topic_coherence'),
            'ARI': results.get('ARI'),
            'AMI': results.get('AMI'),
            'ASW': results.get('ASW'),
            'purity': results.get('purity')
        }
        
        print(f"    ✓ Completed in {elapsed_time/60:.2f} min")
        if results.get('topic_diversity') is not None:
            print(f"      Topic Diversity: {results['topic_diversity']:.4f}")
            print(f"      Topic Coherence: {results['topic_coherence']:.4f}")
        if results.get('ARI') is not None:
            print(f"      ARI: {results['ARI']:.4f}, Purity: {results['purity']:.4f}")
        
        return result_entry
        
    except Exception as e:
        print(f"    ✗ Failed: {e}")
        return {
            'topics': num_topics,
            'seed': seed,
            'use_labels': use_labels,
            'status': 'failed',
            'error': str(e)
        }

# ========== Run experiments ==========
print("=" * 80)
print(f"Stability Analysis: Different Topics and Random Seeds")
print(f"Dataset: {DATASET_NAME}")
print(f"Topics: {TOPICS}")
print(f"Seeds: {SEEDS}")
print(f"Mode: {'WITH labels (ARI, NMI, ASW, Purity)' if USE_LABELS else 'Label-free (no clustering metrics)'}")
print(f"Total experiments: {len(TOPICS) * len(SEEDS)}")
print("=" * 80)

results_list = []

# Determine mode name for output directory
mode_dir = 'with_labels' if USE_LABELS else 'label_free'

for num_topics in TOPICS:
    for seed in SEEDS:
        # Create subdirectory path: topic_{num_topics}/seed_{seed}/{mode_dir}
        subdir = f'topic_{num_topics}/seed_{seed}/{mode_dir}'
        
        result = run_experiment(num_topics, seed, use_labels=USE_LABELS, subdir=subdir)
        results_list.append(result)

# ========== Save summary ==========
print("\n" + "=" * 60)
print("Saving summary...")
print("=" * 60)

summary_df = pd.DataFrame(results_list)
summary_path = BASE_OUTPUT_DIR / f'stability_summary_{mode_dir}.csv'
summary_df.to_csv(summary_path, index=False)

print(f"\nSummary saved to: {summary_path}")
print(f"\nTotal experiments: {len(results_list)}")
print(f"Successful: {len(summary_df[summary_df['status'] == 'success'])}")
print(f"Failed: {len(summary_df[summary_df['status'] == 'failed'])}")

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from itertools import combinations
from pathlib import Path

# ========== Path Configuration ==========
PROJECT_ROOT = Path('/PATH/scE2TM')
DATASET_NAME = 'Wang'
mode_dir = 'with_labels'
BASE_OUTPUT_DIR = PROJECT_ROOT / 'output' / DATASET_NAME

# Topic numbers and seeds
topic_nums = [10, 20, 50, 80, 100]
seeds = [1, 2]  # # quick testing

# ========== Helper Functions ==========
def load_topic_matrix(topic_num, seed):
    """Load topic-gene matrix from CSV file"""
    file_path = BASE_OUTPUT_DIR / f'topic_{topic_num}' / f'seed_{seed}' / mode_dir / f'{DATASET_NAME}_tg.csv'
    
    if not file_path.exists():
        print(f"  File not found: {file_path}")
        return None
    
    try:
        data = pd.read_csv(file_path, index_col=0)
        return data.values  # Return numpy array
    except Exception as e:
        print(f"  Failed to load {file_path}: {e}")
        return None

def calculate_topic_similarity(mat1, mat2):
    """Calculate cosine similarity between two topic matrices"""
    # Normalize rows to unit length
    mat1_norm = normalize(mat1, norm='l2')
    mat2_norm = normalize(mat2, norm='l2')
    similarity = cosine_similarity(mat1_norm, mat2_norm)
    return similarity

def hungarian_matching(similarity_matrix):
    """Perform Hungarian algorithm for topic matching (maximize similarity)"""
    cost_matrix = -similarity_matrix
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    return row_ind, col_ind

def process_single_pair(mat1, mat2):
    """Process a single seed pair and return average similarity after matching"""
    similarity = calculate_topic_similarity(mat1, mat2)
    row_ind, col_ind = hungarian_matching(similarity)
    matched_similarities = similarity[row_ind, col_ind]
    return np.mean(matched_similarities)

# ========== Compute Topic Stability ==========
topic_stability = {}

for topic_num in topic_nums:
    print(f"Processing K={topic_num}...")
    
    # Load matrices for all seeds
    matrices = {}
    valid_seeds = []
    for seed in seeds:
        mat = load_topic_matrix(topic_num, seed)
        if mat is not None:
            matrices[seed] = mat
            valid_seeds.append(seed)
    
    if len(valid_seeds) < 2:
        print(f"  Warning: Less than 2 valid seeds, skipping K={topic_num}")
        topic_stability[topic_num] = np.nan
        continue
    
    # Compute similarity for all seed pairs
    pair_sims = []
    for seed1, seed2 in combinations(valid_seeds, 2):
        sim = process_single_pair(matrices[seed1], matrices[seed2])
        pair_sims.append(sim)
    
    # Average similarity as stability metric
    avg_stability = np.mean(pair_sims)
    topic_stability[topic_num] = avg_stability
    
    print(f"   Stability (avg similarity): {avg_stability:.4f}")

print("\n" + "="*60)
print("Topic Stability Results:")
for topic_num in topic_nums:
    print(f"  K={topic_num}: {topic_stability[topic_num]:.4f}")

# ========== Add to summary_df ==========
# Add stability column to summary_df
summary_df['topic_stability'] = summary_df['topics'].map(topic_stability)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

# ========== Set plot style ==========
plt.style.use('default')
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']
rcParams['font.size'] = 10
rcParams['axes.linewidth'] = 0.5
rcParams['axes.labelsize'] = 11
rcParams['xtick.labelsize'] = 10
rcParams['ytick.labelsize'] = 10
rcParams['legend.fontsize'] = 9
rcParams['figure.dpi'] = 300

# ========== Load data ==========
# Assume summary_df already contains the topic_stability column
# If not, run the previous code to add it first

# Compute mean per topic (across seeds)
metrics = ['ARI', 'AMI', 'topic_coherence', 'topic_diversity', 'topic_stability']
metric_names = {
    'ARI': 'ARI',
    'AMI': 'NMI',
    'topic_coherence': 'TC',
    'topic_diversity': 'TD',
    'topic_stability': 'Stability'
}

# Color scheme
colors = ['#2C78AC', '#F37E34', '#429E48', '#8369AF', '#C779B2']

# Compute mean per topic
topics = sorted(summary_df['topics'].unique())
means = {}

for metric in metrics:
    metric_means = []
    for topic in topics:
        subset = summary_df[summary_df['topics'] == topic]
        mean_val = subset[metric].mean()
        metric_means.append(mean_val)
    means[metric] = np.array(metric_means)

print("Original means per topic:")
for metric in metrics:
    print(f"{metric_names[metric]}: {means[metric]}")

# ========== Normalization: divide each metric by its maximum value ==========
normalized_means = {}
for metric in metrics:
    max_val = np.max(means[metric])
    if max_val > 0:
        normalized_means[metric] = means[metric] / max_val
    else:
        normalized_means[metric] = means[metric]
    print(f"\n{metric_names[metric]} max value: {max_val:.4f}")

# ========== Plot line chart ==========
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(topics))

# Plot lines
lines = []
for i, metric in enumerate(metrics):
    line, = ax.plot(x, normalized_means[metric], 
                    color=colors[i],
                    marker='o',
                    markersize=6,
                    linewidth=1.5,
                    label=metric_names[metric])
    lines.append(line)

# Set x-axis
ax.set_xlabel('Number of Topics', fontsize=11, labelpad=3)
ax.set_ylabel('Normalized Value (0 to max of each metric)', fontsize=11, labelpad=3)

ax.set_xticks(x)
ax.set_xticklabels([str(t) for t in topics], fontsize=10)
ax.tick_params(axis='both', which='major', labelsize=10)

# Set y-axis range
ax.set_ylim(0, 1.05)

# Add grid lines
ax.grid(True, linestyle='--', linewidth=0.3, alpha=0.3, axis='y')
ax.set_axisbelow(True)

# Legend outside to the right
legend = ax.legend(lines, [metric_names[m] for m in metrics],
                   loc='center left',
                   bbox_to_anchor=(1.02, 0.5),
                   frameon=False,
                   handlelength=1.5,
                   handletextpad=0.5,
                   ncol=1,
                   fontsize=9)

# Adjust layout
plt.subplots_adjust(right=0.75)

# Show plot
plt.show()

# ========== Print normalized values table ==========
print("\n" + "="*80)
print("Normalized metrics per topic (divided by max value of each metric)")
print("="*80)

print(f"{'Topics':>8}", end="")
for metric in metrics:
    print(f"{metric_names[metric]:>15}", end="")
print()

for idx, topic in enumerate(topics):
    print(f"{topic:>8}", end="")
    for metric in metrics:
        print(f"{normalized_means[metric][idx]:>15.4f}", end="")
    print()

# ========== Print best topic number for each metric ==========
print("\n" + "="*80)
print("Best topic number for each metric (maximum value)")
print("="*80)
for metric in metrics:
    max_idx = np.argmax(means[metric])
    print(f"{metric_names[metric]}: max value {means[metric][max_idx]:.4f} @ K={topics[max_idx]}")